<a href="https://colab.research.google.com/github/iamrobby/Content-Engine/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.9/499.9 kB 15.9 MB/s eta 0:00:00


In [2]:
!pip install Flask-API gunicorn requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.5/138.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 10.6 MB/s eta 0:00:00


In [3]:
!pip install Flask-RESTful

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.0 MB/s eta 0:00:00


In [41]:
%%writefile settings.py
import os

DEBUG = os.environ.get('DEBUG', True)
SECRET_KEY = os.environ.get('FLASK_SECRET', '1234567890')
API_TOKEN = os.environ.get('API_TOKEN', 'FOOBAR1')
REDIS_URL = os.environ.get('REDIS_URL', 'redis://localhost:6379')

Overwriting settings.py


In [69]:
%%writefile engines.py
import pandas as pd
import time
import redis
from flask import current_app
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

def info(msg):
    current_app.logger.info(msg)

class ContentEngine(object):

    SIMKEY = 'p:smlr:%s'

    def __init__(self):
        self._r = redis.StrictRedis.from_url(current_app.config['REDIS_URL'])

    def train(self, data_source):
        start = time.time()
        ds = pd.read_csv(data_source,on_bad_lines="skip")
        ds.rename(columns=lambda x: x.strip().lower(), inplace=True)

      # Ensure 'description' exists
        if 'description' not in ds.columns:
            ds['description'] = ""

        # Fill missing values
        ds['description'] = ds['description'].fillna("")

        # Convert to string
        ds['description'] = ds['description'].astype(str)
        ds = ds[ds['description'].str.strip() != ""]
        info("Training data ingested in %s seconds." % (time.time() - start))

        # Flush the stale training data from redis
        self._r.flushdb()

        start = time.time()
        self._train(ds)
        info("Engine trained in %s seconds." % (time.time() - start))

    def _train(self, ds):

        tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 3), min_df=0.0,stop_words='english')
        tfidf_matrix = tf.fit_transform(ds['description'])
        print(ds['description'].head(10))

        cosine_similarities = linear_kernel(tfidf_matrix, tfidf_matrix)

        for idx, row in ds.iterrows():
            similar_items = list(enumerate(cosine_similarities[idx]))
            # Get the top 100 similar items, excluding itself
            sorted_similar_items = sorted(similar_items, key=lambda x: x[1], reverse=True)[1:101]

            # Create a dictionary for zadd with member: score format
            # Explicitly convert item ID to string
            zadd_mapping = {str(ds['id'].iloc[i]): float(score) for i, score in sorted_similar_items}

            # Store in Redis using the dictionary mapping
            if zadd_mapping: # Only add if there are items to add
                self._r.zadd(self.SIMKEY % row['id'], zadd_mapping)

    def predict(self, item, num):
        """
        Look up the 'num' most similar items to 'item'

        :param item: Item ID to find similar items for.
        :param num: Number of similar items to return.
        :return: A list of the 'num' most similar items.
        """
        top_similar = self._r.zrevrange(self.SIMKEY % item, 0, num - 1, withscores=True)
        return [(member.decode('utf-8'), score) for member, score in top_similar]

content_engine = ContentEngine()


Overwriting engines.py


In [70]:
%%writefile app.py
from flask import Flask, request, current_app, abort
from flask_restful import Resource, Api
from functools import wraps
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)

app = Flask(__name__)
app.config.from_object('settings')
api = Api(app)

def token_auth(f):
    @wraps(f)
    def decorated_function(*args, **kwargs):
        if request.headers.get('X-API-TOKEN', None) != current_app.config['API_TOKEN']:
            abort(403)
        return f(*args, **kwargs)
    return decorated_function

class Predict(Resource):

    def post(self):
        from engines import content_engine
        item = request.json.get('item') # Flask-RESTful often expects JSON for POST
        num_predictions = request.json.get('num', 10)
        if not item:
            return [], 200 # Return empty list and 200 OK for consistency
        return content_engine.predict(str(item), num_predictions), 200

class Train(Resource):

    def get(self):
        from engines import content_engine
        data_url = request.args.get('data-url', None) # GET requests use request.args
        if data_url is None:
            # Handle case where data-url is not provided
            return {"message": "'data-url' parameter is required for training", "success": 0}, 400
        try:
            content_engine.train(data_url)
            return {"message": "Success!", "success": 1}, 200
        except Exception as e:
            current_app.logger.error(f"Training failed: {e}")
            return {"message": f"Training failed: {e}", "success": 0}, 500

api.add_resource(Predict, '/predict')
api.add_resource(Train, '/train')

if __name__ == '__main__':
    app.debug = True
    app.run(debug=True, host='0.0.0.0', port=5000) # Use 0.0.0.0 for external access in Colab, default Flask port is 5000

Overwriting app.py


In [20]:
# Install Redis server
!apt-get update
!apt-get install -y redis-server

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
redis-server is already the newest version (5:6.0.16-1ubuntu1.1).
0 upgraded, 0 newly installe

In [71]:
import subprocess

# Find processes listening on port 8000 and kill them
try:
    # Find PIDs using port 8000
    lsof_output = subprocess.check_output(['lsof', '-t', '-i', ':8000']).decode().strip()
    if lsof_output:
        pids = lsof_output.split('\n')
        print(f"Found processes on port 8000: {pids}. Attempting to kill them...")
        for pid in pids:
            subprocess.run(['kill', '-9', pid])
        print("Processes killed.")
    else:
        print("No processes found on port 8000.")
except subprocess.CalledProcessError:
    print("No processes found on port 8000.")
except Exception as e:
    print(f"An error occurred: {e}")


Found processes on port 8000: ['26248', '26249']. Attempting to kill them...
Processes killed.


In [72]:
# Start a Redis server in the background, as the Flask app relies on it.
!nohup redis-server --port 6379 &

nohup: appending output to 'nohup.out'


In [73]:
!nohup gunicorn app:app --log-file=- &

nohup: appending output to 'nohup.out'


In [11]:
!pip install pyngrok

In [74]:
import subprocess

print("Attempting to kill all ngrok processes...")
try:
    # Find and kill all ngrok processes
    subprocess.run(['killall', '-9', 'ngrok'], check=True)
    print("All ngrok processes killed successfully.")
except subprocess.CalledProcessError:
    print("No ngrok processes were found or could be killed.")
except Exception as e:
    print(f"An error occurred while trying to kill ngrok processes: {e}")


Attempting to kill all ngrok processes...
All ngrok processes killed successfully.


In [75]:
from pyngrok import ngrok
from google.colab import userdata

# Fetch your ngrok authtoken from Colab Secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

# Authenticate ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open a tunnel to the Flask app running on port 8000
public_url = ngrok.connect(8000)
print(f"Flask app is available at: {public_url}")

Flask app is available at: NgrokTunnel: "https://nonaddicted-pseudonymously-bella.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
# Check the gunicorn logs for any errors
!cat nohup.out

In [ ]:
# payload = {
#     'item': 1, # Assuming you want predictions for item ID 1
#     'num': 5   # Requesting 5 predictions
# }

In [ ]:
# import requests

# response = requests.post(url, json=payload)
# print(response.json())

In [17]:
import pandas as pd
ds = pd.read_csv("netflix_titles.csv" , on_bad_lines="skip")
print(ds.columns.tolist())


['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']
